In [0]:
import sys
from pathlib import Path

# Find the repository root from the notebook's directory
current = Path.cwd()

repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "src" / "house_price_simple_analysis").is_dir()
    ),
    None
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate src/house_price_simple_analysis. "
        f"Current directory: {current}"
    )

sys.path.insert(0, str(repo_root / "src"))

print("Repository root:", repo_root)
print("Python source directory:", repo_root / "src")

In [0]:
import importlib

import house_price_simple_analysis.config as config_module
import house_price_simple_analysis.regression_model as regression_module

# Reload configuration first
importlib.reload(config_module)

# Then reload regression model
importlib.reload(regression_module)

ProjectConfig = config_module.ProjectConfig
RegressionModel = regression_module.RegressionModel

In [0]:
config = ProjectConfig.from_yaml(
    str(repo_root / "config" / "project_config.yml")
)

regression = RegressionModel(
    config=config,
    spark=spark
)


print(config.model.registered_model_name)

### Testing function Load Data

In [0]:
# 1. Load data
regression.load_data()

print("Training dataset:", regression.train_set.shape)
print("Testing dataset:", regression.test_set.shape)

In [0]:
regression.train_set.head()

In [0]:
regression.test_set.head()

### Prepare the model

In [0]:
# 2. Prepare model
regression.prepare_model()

### Training the model

In [0]:
# 3. Train model
regression.train()

### Evaluate the model

In [0]:
# 4. Evaluate model
metrics = regression.evaluate()

print("Evaluation metrics:", metrics)

### logging model using mlflow

In [0]:
# 5. Log to MLflow
model_uri = regression.log_model()

print("MLflow model URI:", model_uri)

### Testing register model

You can choose the model you want to register by manually adding the model's URL; if you don't, the function will automatically add the URL generated after the last time you logged the model.

In [0]:
regression.model_uri = "models:/m-c68f3c7d426b4f619809a2da79c55888"

In [0]:
# Register the model
registered_uri = regression.register_model()

print("Registered model:", regression.registered_model_name)
print("Version:", regression.registered_model_version)
print("URI:", registered_uri)

### Retrieving the registed model

In [0]:
import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

loaded_model = mlflow.sklearn.load_model(registered_uri)

sample = pd.DataFrame({
    "square_footage": [2000.0]
})

prediction = loaded_model.predict(sample)

print("Predicted price:", prediction[0])